# TRI Tracker Update

This notebook computes tariff metrics and prepares the data file used by the TRI Tracker app.
Run this notebook when you only need to update the tracker, without regenerating paper results.

In [1]:
import pandas as pd
import numpy as np
import os

import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
data_repo = "C:\\heroku\\how-restrictive-us-trade\\data\\"

country_list = pd.read_csv(os.path.join(data_repo, "country-list.csv"), header=None, names=["CTY_CODE"])

target_date = "2026-06"

In [3]:
def make_tariff_country_date(dfcntry, date):
   
   bar = dfcntry[(dfcntry['time'] == date)].copy()

   bar["tariff"] = bar["CAL_DUT_MO"] / bar["CON_VAL_MO"]

   bar = bar[["I_COMMODITY", "CON_VAL_MO", "CAL_DUT_MO", "tariff", "CTY_NAME"]]

   return bar

In [4]:
def make_good_country_year(dfcntry, year):
    """
    Process country data for a specific year and calculate aggregate metrics.
    """
    df_year = dfcntry[dfcntry['time']== year]
    
    grp = df_year.groupby("I_COMMODITY")
    
    result = grp.agg({
        "CON_VAL_MO": "sum",
        "CAL_DUT_MO": "sum",
        "CTY_NAME": "first"})
    
    return result

In [5]:
def process_countries_for_date(country_list, target_date, weight_year="2024", selected_country=None):
    """
    Process multiple countries and create combined dataset with weights.
    """
    if isinstance(country_list, pd.DataFrame):
        countries = country_list.CTY_CODE
    else:
        countries = country_list
    
    all_results = []
    tariff_results = []
    
    for country in countries:
        
        file = os.path.join(data_repo, f"imports-hs10\\{country}data-current.parquet")
        
        dfcntry = pq.read_table(file).to_pandas()
        
        dfcntry["CAL_DUT_MO"] = dfcntry["CAL_DUT_MO"].astype(float)
        dfcntry["CON_VAL_MO"] = dfcntry["CON_VAL_MO"].astype(float)
        dfcntry.time = pd.to_datetime(dfcntry.time, format="%Y-%m")
        dfcntry["applied_tariff"] = dfcntry["CAL_DUT_MO"] / dfcntry["CON_VAL_MO"]
            
        results_df = make_good_country_year(dfcntry, weight_year)
        results_df.reset_index(inplace=True)
        
        tariff_df = make_tariff_country_date(dfcntry, target_date)
        tariff_df.reset_index(drop=True, inplace=True)
        
        all_results.append(results_df)
        tariff_results.append(tariff_df)
        
        print(f"Processed {country}: {results_df['CTY_NAME'].iloc[0]}")
    
    combined_results = pd.concat(all_results)
    combined_results.reset_index(drop=True, inplace=True)
    
    combined_tariff_results = pd.concat(tariff_results)
    combined_tariff_results.reset_index(drop=True, inplace=True)
    
    combined_results["weights"] = combined_results["CON_VAL_MO"] / combined_results["CON_VAL_MO"].sum()
    
    bigdf = pd.merge(
        combined_results[["I_COMMODITY", "CTY_NAME", "weights"]], 
        combined_tariff_results, 
        how="left", 
        on=["I_COMMODITY", "CTY_NAME"]
    )
    
    if selected_country is not None:
        bigdf = bigdf[bigdf["CTY_NAME"] == selected_country].copy()
        bigdf["weights"] = bigdf["weights"] / bigdf["weights"].sum()
    
    return bigdf

In [6]:
def create_tariff_metrics_by_date_country(country_list, date_list, weight_year="2024"):
    """
    Calculate tariff metrics for multiple dates and countries.
    """
    results = []
    
    for date in date_list:
        print(f"Processing date: {date}")
        
        bigdf = process_countries_for_date(country_list, date, weight_year=weight_year)
        
        sqrtariff_all = ((bigdf["tariff"]**2 * bigdf["weights"]).sum())**0.5
        meanweighted_all = (bigdf["tariff"] * bigdf["weights"]).sum()
        simplemean_all = bigdf['CAL_DUT_MO'].sum() / bigdf["CON_VAL_MO"].sum()
        
        results.append({
            'date': date,
            'CTY_NAME': 'ALL COUNTRIES',
            'sqrtariff': sqrtariff_all,
            'meanweighted': meanweighted_all,
            'simplemean': simplemean_all,
            "duty_total": bigdf['CAL_DUT_MO'].sum()
        })
        
        countries = bigdf['CTY_NAME'].unique()
        
        for country in countries:
            country_df = bigdf[bigdf['CTY_NAME'] == country].copy()
            country_df["weights"] = country_df["weights"] / country_df["weights"].sum()
            
            sqrtariff = ((country_df["tariff"]**2 * country_df["weights"]).sum())**0.5
            meanweighted = (country_df["tariff"] * country_df["weights"]).sum()
            simplemean = country_df['CAL_DUT_MO'].sum() / country_df["CON_VAL_MO"].sum()
            
            results.append({
                'date': date,
                'CTY_NAME': country,
                'sqrtariff': sqrtariff,
                'meanweighted': meanweighted,
                'simplemean': simplemean,
                "duty_total": country_df['CAL_DUT_MO'].sum()
            })
    
    results_df = pd.DataFrame(results)
    return results_df

In [7]:
date_list = pd.date_range(start="2024-01", end=target_date, freq="MS").strftime("%Y-%m").tolist()

metrics_df = create_tariff_metrics_by_date_country(country_list, date_list, weight_year="2024")

metrics_df["date"] = pd.to_datetime(metrics_df["date"], format="%Y-%m")

metrics_df.to_parquet(data_repo + "\\top-country-metrics.parquet", index=False)

metrics_df = pq.read_table(data_repo + "\\top-country-metrics.parquet").to_pandas()

Processing date: 2024-01
Processed 5700: CHINA
Processed 2010: MEXICO
Processed 1220: CANADA
Processed 5880: JAPAN
Processed 4280: GERMANY
Processed 5800: KOREA, SOUTH
Processed 5520: VIETNAM
Processed 5830: TAIWAN
Processed 4190: IRELAND
Processed 5330: INDIA
Processed 4120: UNITED KINGDOM
Processed 4759: ITALY
Processed 4279: FRANCE
Processed 4419: SWITZERLAND
Processed 5570: MALAYSIA
Processed 5490: THAILAND
Processed 3510: BRAZIL
Processed 5590: SINGAPORE
Processed 4210: NETHERLANDS
Processed 5600: INDONESIA
Processed 5081: ISRAEL
Processed 4231: BELGIUM
Processed 5170: SAUDI ARABIA
Processed 4700: SPAIN
Processed 4621: RUSSIA
Processed 3010: COLOMBIA
Processed 4330: AUSTRIA
Processed 6021: AUSTRALIA
Processed 4010: SWEDEN
Processed 3370: CHILE
Processing date: 2024-02
Processed 5700: CHINA
Processed 2010: MEXICO
Processed 1220: CANADA
Processed 5880: JAPAN
Processed 4280: GERMANY
Processed 5800: KOREA, SOUTH
Processed 5520: VIETNAM
Processed 5830: TAIWAN
Processed 4190: IRELAND
Pr

C:\Users\micha\AppData\Local\Temp\ipykernel_6960\4066074239.py:33: RuntimeWarning: invalid value encountered in scalar divide
  simplemean = country_df['CAL_DUT_MO'].sum() / country_df["CON_VAL_MO"].sum()


Processing date: 2026-05
Processed 5700: CHINA
Processed 2010: MEXICO
Processed 1220: CANADA
Processed 5880: JAPAN
Processed 4280: GERMANY
Processed 5800: KOREA, SOUTH
Processed 5520: VIETNAM
Processed 5830: TAIWAN
Processed 4190: IRELAND
Processed 5330: INDIA
Processed 4120: UNITED KINGDOM
Processed 4759: ITALY
Processed 4279: FRANCE
Processed 4419: SWITZERLAND
Processed 5570: MALAYSIA
Processed 5490: THAILAND
Processed 3510: BRAZIL
Processed 5590: SINGAPORE
Processed 4210: NETHERLANDS
Processed 5600: INDONESIA
Processed 5081: ISRAEL
Processed 4231: BELGIUM
Processed 5170: SAUDI ARABIA
Processed 4700: SPAIN
Processed 4621: RUSSIA
Processed 3010: COLOMBIA
Processed 4330: AUSTRIA
Processed 6021: AUSTRALIA
Processed 4010: SWEDEN
Processed 3370: CHILE


C:\Users\micha\AppData\Local\Temp\ipykernel_6960\4066074239.py:33: RuntimeWarning: invalid value encountered in scalar divide
  simplemean = country_df['CAL_DUT_MO'].sum() / country_df["CON_VAL_MO"].sum()


Processing date: 2026-06
Processed 5700: CHINA
Processed 2010: MEXICO
Processed 1220: CANADA
Processed 5880: JAPAN
Processed 4280: GERMANY
Processed 5800: KOREA, SOUTH
Processed 5520: VIETNAM
Processed 5830: TAIWAN
Processed 4190: IRELAND
Processed 5330: INDIA
Processed 4120: UNITED KINGDOM
Processed 4759: ITALY
Processed 4279: FRANCE
Processed 4419: SWITZERLAND
Processed 5570: MALAYSIA
Processed 5490: THAILAND
Processed 3510: BRAZIL
Processed 5590: SINGAPORE
Processed 4210: NETHERLANDS
Processed 5600: INDONESIA
Processed 5081: ISRAEL
Processed 4231: BELGIUM
Processed 5170: SAUDI ARABIA
Processed 4700: SPAIN
Processed 4621: RUSSIA
Processed 3010: COLOMBIA
Processed 4330: AUSTRIA
Processed 6021: AUSTRALIA
Processed 4010: SWEDEN
Processed 3370: CHILE


C:\Users\micha\AppData\Local\Temp\ipykernel_6960\4066074239.py:33: RuntimeWarning: invalid value encountered in scalar divide
  simplemean = country_df['CAL_DUT_MO'].sum() / country_df["CON_VAL_MO"].sum()


In [8]:
# this loads the statatory tariffs data

stat_tariffs = pd.read_csv('C:\\github\\trade-war-redux-2025\\country-by-time.csv')

stat_tariffs.date = pd.to_datetime(stat_tariffs.date, format="%Y-%m-%d")

stat_tariffs.rename(columns = {"country_name": "CTY_NAME"}, inplace = True)

stat_tariffs["effective tariff"] = 0.01*stat_tariffs["effective tariff"] + 0.01*stat_tariffs["2024 tariff"]

daily_tariff = pd.read_csv("C:\\github\\trade-war-redux-2025\\daily-tariff-latest-data.csv", usecols=["date", "import_weighted_avg_tariff"])

daily_tariff["date"] = pd.to_datetime(daily_tariff["date"])

daily_tariff["CTY_NAME"] = "ALL COUNTRIES"

daily_tariff["import_weighted_avg_tariff"] = 0.01*daily_tariff["import_weighted_avg_tariff"]

daily_tariff.rename(columns = {"import_weighted_avg_tariff":"effective tariff"}, inplace=True)

stat_tariffs = pd.concat([stat_tariffs, daily_tariff], ignore_index=True)

In [9]:
stat_tariffs.tail()

,date,CTY_NAME,effective tariff,total imports,2024 tariff
1798,2026-09-18,ALL COUNTRIES,0.114567,NaN,NaN
1799,2026-09-19,ALL COUNTRIES,0.114567,NaN,NaN
1800,2026-09-20,ALL COUNTRIES,0.114567,NaN,NaN
1801,2026-09-21,ALL COUNTRIES,0.114567,NaN,NaN
1802,2026-09-22,ALL COUNTRIES,0.114567,NaN,NaN


In [10]:
# Country to flag URL mapping

country_flags = {
    'ALL COUNTRIES': 'https://em-content.zobj.net/thumbs/120/twitter/348/globe-showing-americas_1f30e.png',
    'CHINA': 'https://flagcdn.com/w40/cn.png',
    'JAPAN': 'https://flagcdn.com/w40/jp.png',
    'MEXICO': 'https://flagcdn.com/w40/mx.png',
    'CANADA': 'https://flagcdn.com/w40/ca.png',
    'VIETNAM': 'https://flagcdn.com/w40/vn.png',
    'KOREA, SOUTH': 'https://flagcdn.com/w40/kr.png',
    'INDIA': 'https://flagcdn.com/w40/in.png',
    'TAIWAN': 'https://flagcdn.com/w40/tw.png',
    'GERMANY': 'https://flagcdn.com/w40/de.png',
    'UNITED KINGDOM': 'https://flagcdn.com/w40/gb.png',
    'FRANCE': 'https://flagcdn.com/w40/fr.png',
    'ITALY': 'https://flagcdn.com/w40/it.png',
    'IRELAND': 'https://flagcdn.com/w40/ie.png',
    'SWITZERLAND': 'https://flagcdn.com/w40/ch.png',
    'THAILAND': 'https://flagcdn.com/w40/th.png',
    'NETHERLANDS': 'https://flagcdn.com/w40/nl.png',
    'BRAZIL': 'https://flagcdn.com/w40/br.png',
    'BELGIUM': 'https://flagcdn.com/w40/be.png',
    'SINGAPORE': 'https://flagcdn.com/w40/sg.png',
    'INDONESIA': 'https://flagcdn.com/w40/id.png',
    'MALAYSIA': 'https://flagcdn.com/w40/my.png',
    'ISRAEL': 'https://flagcdn.com/w40/il.png',
    'SAUDI ARABIA': 'https://flagcdn.com/w40/sa.png',
    'SPAIN': 'https://flagcdn.com/w40/es.png',
    'RUSSIA': 'https://flagcdn.com/w40/ru.png',
    'COLOMBIA': 'https://flagcdn.com/w40/co.png',
    'AUSTRIA': 'https://flagcdn.com/w40/at.png',
    'AUSTRALIA': 'https://flagcdn.com/w40/au.png',
    'SWEDEN': 'https://flagcdn.com/w40/se.png',
    'CHILE': 'https://flagcdn.com/w40/cl.png'
}

# Country to color mapping (using national/flag colors)
country_colors = {
    'ALL COUNTRIES': '#000000',
    'CHINA': '#DE2910',
    'JAPAN': '#BC002D',
    'MEXICO': '#006847',
    'CANADA': '#FF0000',
    'VIETNAM': '#DA251D',
    'KOREA, SOUTH': '#003478',
    'INDIA': '#FF9933',
    'TAIWAN': '#000095',
    'GERMANY': '#FFCE00',
    'UNITED KINGDOM': '#012169',
    'FRANCE': '#0055A4',
    'ITALY': '#009246',
    'IRELAND': '#169B62',
    'SWITZERLAND': '#FF0000',
    'THAILAND': '#2D2A4A',
    'NETHERLANDS': '#FF6600',
    'BRAZIL': '#009B3A',
    'BELGIUM': '#FDDA24',
    'SINGAPORE': '#EE2737',
    'INDONESIA': '#FF0000',
    'MALAYSIA': '#CC0001',
    'ISRAEL': '#0038B8',
    'SAUDI ARABIA': '#165B33',
    'SPAIN': '#AA151B',
    'RUSSIA': '#0039A6',
    'COLOMBIA': '#FCD116',
    'AUSTRIA': '#ED2939',
    'AUSTRALIA': '#012169',
    'SWEDEN': '#006AA7',
    'CHILE': '#D52B1E'
}

# Add flag and color columns
metrics_df['flag'] = metrics_df['CTY_NAME'].map(country_flags)
metrics_df['color'] = metrics_df['CTY_NAME'].map(country_colors)

# Create complete date range — extend to cover all stat_tariffs dates
min_date = metrics_df['date'].min()
max_date = max(pd.to_datetime(target_date, format="%Y-%m") + pd.offsets.MonthEnd(2), stat_tariffs['date'].max())
full_date_range = pd.date_range(start=min_date, end=max_date, freq='D')

# End of the last observed month
last_obs_month_end = metrics_df['date'].max() + pd.offsets.MonthEnd(0)

# Resample to daily frequency with forward fill (not interpolation)
daily_data = []

for country in metrics_df['CTY_NAME'].unique():

    country_df = metrics_df[metrics_df['CTY_NAME'] == country].copy()
    country_df = country_df.set_index('date').sort_index()

    # Remove duplicate dates by keeping the last occurrence
    country_df = country_df[~country_df.index.duplicated(keep='last')]

    # Resample to daily and forward fill (each day gets that month's value)
    country_daily = country_df[['sqrtariff', 'meanweighted', 'simplemean', 'duty_total']].resample('D').ffill()

    # Reindex to full date range and forward fill through end of last observed month
    country_daily = country_daily.reindex(full_date_range).ffill()

    # Null out tariff metrics beyond end of last observed month
    country_daily.loc[country_daily.index > last_obs_month_end, ['sqrtariff', 'meanweighted', 'simplemean', 'duty_total']] = np.nan

    # Add back the country name, flag, and color (constant for each country)
    country_daily['CTY_NAME'] = country
    country_daily['flag'] = country_df['flag'].iloc[0]
    country_daily['color'] = country_df['color'].iloc[0]

    country_daily = country_daily.reset_index()
    country_daily.rename(columns={'index': 'date'}, inplace=True)
    daily_data.append(country_daily)

# Combine all countries
app_data = pd.concat(daily_data, ignore_index=True)

app_data = app_data[['date', 'CTY_NAME', 'sqrtariff', 'meanweighted', 'simplemean', 'duty_total', 'flag', 'color']]

app_data = pd.merge(stat_tariffs[["CTY_NAME", "date", "effective tariff"]], app_data, how="right", on=["date", "CTY_NAME"])

# Fill missing effective tariff values with EUROPEAN UNION values for countries not in stat_tariffs
eu_tariffs = stat_tariffs[stat_tariffs["CTY_NAME"] == "EUROPEAN UNION"][["date", "effective tariff"]].copy()
eu_tariffs.rename(columns={"effective tariff": "eu_effective_tariff"}, inplace=True)
app_data = pd.merge(app_data, eu_tariffs, how="left", on="date")
app_data["effective tariff"] = app_data["effective tariff"].fillna(app_data["eu_effective_tariff"])
app_data.drop(columns=["eu_effective_tariff"], inplace=True)

# Backfill effective tariff with meanweighted for dates before 2025-02-04
app_data.loc[app_data['date'] < '2025-02-04', 'effective tariff'] = app_data.loc[app_data['date'] < '2025-02-04', 'meanweighted']

# Forward fill effective tariff to propagate values
app_data['effective tariff'] = app_data.groupby('CTY_NAME')['effective tariff'].ffill()

app_data.to_parquet('../TRI-tracker/data/tri-all-country-data.parquet')

In [11]:
app_data

,CTY_NAME,date,effective tariff,sqrtariff,meanweighted,simplemean,duty_total,flag,color
0,ALL COUNTRIES,2024-01-01,0.025379,0.069033,0.025379,0.025379,5.842121e+09,https://em-content.zobj.net/thumbs/120/twitter...,#000000
1,ALL COUNTRIES,2024-01-02,0.025379,0.069033,0.025379,0.025379,5.842121e+09,https://em-content.zobj.net/thumbs/120/twitter...,#000000
2,ALL COUNTRIES,2024-01-03,0.025379,0.069033,0.025379,0.025379,5.842121e+09,https://em-content.zobj.net/thumbs/120/twitter...,#000000
3,ALL COUNTRIES,2024-01-04,0.025379,0.069033,0.025379,0.025379,5.842121e+09,https://em-content.zobj.net/thumbs/120/twitter...,#000000
4,ALL COUNTRIES,2024-01-05,0.025379,0.069033,0.025379,0.025379,5.842121e+09,https://em-content.zobj.net/thumbs/120/twitter...,#000000
...,...,...,...,...,...,...,...,...,...
30871,CHILE,2026-09-18,0.072841,NaN,NaN,NaN,NaN,https://flagcdn.com/w40/cl.png,#D52B1E
30872,CHILE,2026-09-19,0.072841,NaN,NaN,NaN,NaN,https://flagcdn.com/w40/cl.png,#D52B1E
30873,CHILE,2026-09-20,0.072841,NaN,NaN,NaN,NaN,https://flagcdn.com/w40/cl.png,#D52B1E
30874,CHILE,2026-09-21,0.072841,NaN,NaN,NaN,NaN,https://flagcdn.com/w40/cl.png,#D52B1E


In [12]:
app_data[(app_data.CTY_NAME == "GERMANY") & (app_data.date == "2026-04-01")]

,CTY_NAME,date,effective tariff,sqrtariff,meanweighted,simplemean,duty_total,flag,color
5801,GERMANY,2026-04-01,0.096198,0.13787,0.097819,0.088632,1.169317e+09,https://flagcdn.com/w40/de.png,#FFCE00
